In [1]:
# Contruct a two-layer GNN model
import dgl.nn as dglnn
import torch.nn as nn
import torch.nn.functional as F
class SAGE(nn.Module):
    def __init__(self, in_feats, hid_feats, out_feats):
        super().__init__()
        self.conv1 = dglnn.SAGEConv(
            in_feats=in_feats, out_feats=hid_feats, aggregator_type='mean')
        self.conv2 = dglnn.SAGEConv(
            in_feats=hid_feats, out_feats=out_feats, aggregator_type='mean')

    def forward(self, graph, inputs):
        # inputs are features of nodes
        h = self.conv1(graph, inputs)
        h = F.relu(h)
        h = self.conv2(graph, h)
        return h

c:\Users\jaspe\Documents\PyEnvs\thesis_py311_v2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dgl.data import CiteseerGraphDataset
dataset = CiteseerGraphDataset()
graph = dataset[0]
print(graph)

C:\Users\jaspe\.dgl\citeseer.zip: 100%|██████████| 239k/239k [00:00<00:00, 485kB/s] 


Extracting file to C:\Users\jaspe\.dgl\citeseer_d6836239
Finished data loading and preprocessing.
  NumNodes: 3327
  NumEdges: 9228
  NumFeats: 3703
  NumClasses: 6
  NumTrainingSamples: 120
  NumValidationSamples: 500
  NumTestSamples: 1000
Done saving data into cached files.
Graph(num_nodes=3327, num_edges=9228,
      ndata_schemes={'train_mask': Scheme(shape=(), dtype=torch.bool), 'val_mask': Scheme(shape=(), dtype=torch.bool), 'test_mask': Scheme(shape=(), dtype=torch.bool), 'label': Scheme(shape=(), dtype=torch.int64), 'feat': Scheme(shape=(3703,), dtype=torch.float32)}
      edata_schemes={})


In [4]:
node_features = graph.ndata['feat']
node_labels = graph.ndata['label']
train_mask = graph.ndata['train_mask']
valid_mask = graph.ndata['val_mask']
test_mask = graph.ndata['test_mask']
n_features = node_features.shape[1]
n_labels = int(node_labels.max().item() + 1)

In [8]:
node_features

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [9]:
node_features.shape 

torch.Size([3327, 3703])

In [10]:
def evaluate(model, graph, features, labels, mask):
    model.eval()
    with torch.no_grad():
        logits = model(graph, features)
        logits = logits[mask]
        labels = labels[mask]
        _, indices = torch.max(logits, dim=1)
        correct = torch.sum(indices == labels)
        return correct.item() * 1.0 / len(labels)

In [11]:
import torch

model = SAGE(in_feats=n_features, hid_feats=100, out_feats=n_labels)
opt = torch.optim.Adam(model.parameters())

for epoch in range(180):
    model.train()
    # forward propagation by using all nodes
    logits = model(graph, node_features)
    # compute loss
    loss = F.cross_entropy(logits[train_mask], node_labels[train_mask])
    # compute validation accuracy
    acc = evaluate(model, graph, node_features, node_labels, valid_mask)
    # backward propagation
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 5 == 0:
        print(f'Epoch {epoch:05d} | Loss {loss.item():.4f} | '
              f'Val. Accuracy {acc:.4f}')

    # Save model if necessary.  Omitted in this example.

Epoch 00000 | Loss 1.7927 | Val. Accuracy 0.2140
Epoch 00005 | Loss 1.7255 | Val. Accuracy 0.3320
Epoch 00010 | Loss 1.6438 | Val. Accuracy 0.3940
Epoch 00015 | Loss 1.5472 | Val. Accuracy 0.4560
Epoch 00020 | Loss 1.4380 | Val. Accuracy 0.4880
Epoch 00025 | Loss 1.3203 | Val. Accuracy 0.5220
Epoch 00030 | Loss 1.1977 | Val. Accuracy 0.5320
Epoch 00035 | Loss 1.0734 | Val. Accuracy 0.5620
Epoch 00040 | Loss 0.9510 | Val. Accuracy 0.5860
Epoch 00045 | Loss 0.8335 | Val. Accuracy 0.6180
Epoch 00050 | Loss 0.7236 | Val. Accuracy 0.6320
Epoch 00055 | Loss 0.6229 | Val. Accuracy 0.6380
Epoch 00060 | Loss 0.5329 | Val. Accuracy 0.6540
Epoch 00065 | Loss 0.4538 | Val. Accuracy 0.6520
Epoch 00070 | Loss 0.3856 | Val. Accuracy 0.6560
Epoch 00075 | Loss 0.3277 | Val. Accuracy 0.6600
Epoch 00080 | Loss 0.2791 | Val. Accuracy 0.6600
Epoch 00085 | Loss 0.2385 | Val. Accuracy 0.6580
Epoch 00090 | Loss 0.2049 | Val. Accuracy 0.6600
Epoch 00095 | Loss 0.1771 | Val. Accuracy 0.6620
Epoch 00100 | Loss 0

In [12]:
g = graph
devide = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
g = g.to(devide)

### Edge classification

In [15]:
import numpy as np
import dgl

src = np.random.randint(0, 100, 500)
dst = np.random.randint(0, 100, 500)
# make it symmetric
edge_pred_graph = dgl.graph((np.concatenate([src, dst]), np.concatenate([dst, src])))
# synthetic node and edge features, as well as edge labels
edge_pred_graph.ndata['feature'] = torch.randn(100, 10)
edge_pred_graph.edata['feature'] = torch.randn(1000, 10)
edge_pred_graph.edata['label'] = torch.randn(1000)
# synthetic train-validation-test splits
edge_pred_graph.edata['train_mask'] = torch.zeros(1000, dtype=torch.bool).bernoulli(0.6)

In [16]:
import dgl.function as fn
class DotProductPredictor(nn.Module):
    def forward(self, graph, h):
        # h contains the node representations computed from the GNN defined
        # in the node classification section (Section 5.1).
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(fn.u_dot_v('h', 'h', 'score'))
            return graph.edata['score']

In [17]:
class MLPPredictor(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2, out_classes)

    def apply_edges(self, edges):
        h_u = edges.src['h']
        h_v = edges.dst['h']
        score = self.W(torch.cat([h_u, h_v], 1))
        return {'score': score}

    def forward(self, graph, h):
        # h contains the node representations computed from the GNN defined
        # in the node classification section (Section 5.1).
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(self.apply_edges)
            return graph.edata['score']

In [18]:
class Model(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.sage = SAGE(in_features, hidden_features, out_features)
        self.pred = DotProductPredictor()
    def forward(self, g, x):
        h = self.sage(g, x)
        return self.pred(g, h)

In [20]:
node_features = edge_pred_graph.ndata['feature']
edge_label = edge_pred_graph.edata['label']
train_mask = edge_pred_graph.edata['train_mask']

model = Model(10, 20, 5)
opt = torch.optim.Adam(model.parameters())

for epoch in range(500):
    pred = model(edge_pred_graph, node_features)
    loss = ((pred[train_mask] - edge_label[train_mask]) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 5 == 0:
        print(f'Epoch {epoch:05d} | Loss {loss.item():.4f}')

Epoch 00000 | Loss 54.3563
Epoch 00005 | Loss 35.8558
Epoch 00010 | Loss 24.1841
Epoch 00015 | Loss 17.2072
Epoch 00020 | Loss 13.0927
Epoch 00025 | Loss 10.5841
Epoch 00030 | Loss 8.9506
Epoch 00035 | Loss 7.8026
Epoch 00040 | Loss 6.9406
Epoch 00045 | Loss 6.2579
Epoch 00050 | Loss 5.7018
Epoch 00055 | Loss 5.2380
Epoch 00060 | Loss 4.8456
Epoch 00065 | Loss 4.5105
Epoch 00070 | Loss 4.2215
Epoch 00075 | Loss 3.9691
Epoch 00080 | Loss 3.7478
Epoch 00085 | Loss 3.5530
Epoch 00090 | Loss 3.3798
Epoch 00095 | Loss 3.2247
Epoch 00100 | Loss 3.0857
Epoch 00105 | Loss 2.9603
Epoch 00110 | Loss 2.8468
Epoch 00115 | Loss 2.7448
Epoch 00120 | Loss 2.6516
Epoch 00125 | Loss 2.5656
Epoch 00130 | Loss 2.4864
Epoch 00135 | Loss 2.4133
Epoch 00140 | Loss 2.3457
Epoch 00145 | Loss 2.2831
Epoch 00150 | Loss 2.2251
Epoch 00155 | Loss 2.1712
Epoch 00160 | Loss 2.1211
Epoch 00165 | Loss 2.0742
Epoch 00170 | Loss 2.0304
Epoch 00175 | Loss 1.9893
Epoch 00180 | Loss 1.9508
Epoch 00185 | Loss 1.9144
Epoch 